<a href="https://colab.research.google.com/github/tsakailab/MultivariateAnalysis/blob/main/ipynb/ex_PolynomialRegression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 例題で理解する回帰分析（多項式回帰と正則化）

* モデル変数 $\tilde{\boldsymbol{\beta}}$ の線形回帰モデルによる予測の誤差（残差）$\boldsymbol{\delta}$ は，
標本（データ）で決まる計画行列 $\tilde{\boldsymbol{X}}\in\mathbb{R}^{n\times (p+1)}$ と $\boldsymbol{y}\in\mathbb{R}^n$ を用いて
$$\boldsymbol{\delta}=\boldsymbol{y}-\tilde{\boldsymbol{X}}\tilde{\boldsymbol{\beta}}$$
と定義されます．
    * 説明変数の数が $p=1$ のときは線形単回帰，$p>1$ のときは線形重回帰です．

* **Ridge回帰（$\ell_2$正則化）**  
[残差平方和](https://en.wikipedia.org/wiki/Residual_sum_of_squares) $\|\boldsymbol{\delta}\|_2^2=\sum_{i=1}^n\delta_i$ を，回帰係数の二乗和 $\|\boldsymbol{\beta}\|_2^2=\sum_{j=1}^p\beta_j^2$ とともに最小にするモデル変数 $$\tilde{\boldsymbol{\beta}}^\star=\left[\begin{array}{c}\beta_0^\star\\ \boldsymbol{\beta}^\star\end{array}\right]=\arg\min_{\tilde{\boldsymbol{\beta}}}\left\{\|\boldsymbol{y}-\tilde{\boldsymbol{X}}\tilde{\boldsymbol{\beta}}\|_2^2+\lambda\|\boldsymbol{\beta}\|_2^2\right\}\quad\in\mathbb{R}^{p+1}$$ は，標本平均および中心化した標本を用いると，次式のようにのように計算できます．
    * 切片　　　$\beta_0^\star=\bar{y}-{\boldsymbol{\beta}^\star}^\top\bar{\boldsymbol{x}}\qquad\in\mathbb{R}$  
        回帰係数　$\boldsymbol{\beta}^\star=(\boldsymbol{X}_c^\top\boldsymbol{X}_c+\lambda\boldsymbol{I}_p)^{-1}\boldsymbol{X}_c^\top\boldsymbol{y}_c\quad\in\mathbb{R}^p$  
        * 目的変数の標本平均： $\bar{y}=\frac{1}{n}\sum_{i=1}^ny^{(i)}$
        * 説明変数の標本平均： $\bar{\boldsymbol{x}}\in\mathbb{R}^p$
        * 中心化したデータフレーム： $\boldsymbol{X}_c\in\mathbb{R}^{n\times p}$
        * 正則化係数： $\lambda>0$，　　$p$次単位行列： $\boldsymbol{I}_p\in\mathbb{R}^{p\times p}$
    * $\lambda\rightarrow 0$のとき，通常の最小二乗解 $\tilde{\boldsymbol{\beta}}^\star\rightarrow (\tilde{\boldsymbol{X}}^\top\tilde{\boldsymbol{X}})^{-1}\tilde{\boldsymbol{X}}^\top\boldsymbol{y}$ と一致する．

---
# 多項式回帰（$p$次の場合）

モデル: $f_{\tilde{\boldsymbol{\beta}}}(x) = \beta_0 + \beta_1 x + \beta_2 x^2+\cdots+\beta_px^p$

説明変数 $x$ と目的変数 $y$ の関係が，直線ではなく曲線を描くと考えられる場合に使用します．例えば，「最初は効果が上がるが，だんだん効果が薄れ，むしろ逆効果が現れる」，「加速度的に上昇する」，「一時的に増減が変わる」といった現象を表現できます．

## 計画行列
計画行列は
$$\tilde{\boldsymbol{X}}=\begin{bmatrix}
1 & x^{(1)} & {x^{(1)}}^2 & \dots & {x^{(1)}}^p \\
1 & x^{(2)} & {x^{(2)}}^2 & \dots & {x^{(2)}}^p \\
\vdots & \vdots & \vdots & \ddots & \vdots \\
1 & x^{(n)} & {x^{(n)}}^2 & \dots & {x^{(n)}}^p \end{bmatrix}\quad\in\mathbb{R}^{n\times (p+1)}$$
ですが，べき乗による要素は桁違いに異なる数値になりやすく，低次または高次の値ほど極端に重視・軽視される偏った分析になる不都合が生じやすいです．
この対策として，多項式特徴量を生成する前に元のデータを標準化するのが一般的です．

## 標準化
$x^{(1)},\dots,x^{(n)}$ を標準化（＝中心化と正規化）した $x_\mathrm{s}^{(1)},\dots,x_\mathrm{s}^{(n)}$ による
$$\boldsymbol{X}_c=\begin{bmatrix}
x_\mathrm{s}^{(1)} & {x_\mathrm{s}^{(1)}}^2 & \dots & {x_\mathrm{s}^{(1)}}^p \\
x_\mathrm{s}^{(2)} & {x_\mathrm{s}^{(2)}}^2 & \dots & {x_\mathrm{s}^{(2)}}^p \\
\vdots & \vdots & \ddots & \vdots \\
x_\mathrm{s}^{(n)} & {x_\mathrm{s}^{(n)}}^2 & \dots & {x_\mathrm{s}^{(n)}}^p \end{bmatrix}\quad\in\mathbb{R}^{n\times p}$$
を用います．

* スケールの違いの是正：元の $x^{(i)}$ の値が大きい場合（例: 1000），2乗すると100万，3乗すると10億となり，特徴量間で値のスケールが極端に異なってしまいます．
* 最適化アルゴリズムの効率化：スケールの異なる特徴量が混在すると，最適化に時間がかかったり，数値計算が不安定になる可能性があります．
* 多重共線性の影響軽減：標準化しないとき，高次の項どうしが強い相関を持ちやすいです．標準化はこれを完全に解消はしないものの，影響を軽減するのに役立ちます．
* 正則化の効果の均一化：ridge回帰やLASSO回帰といった正則化手法を用いる場合，標準化によって各特徴量に公平にペナルティが課されるようになります．

---
## 例題：肥料の量と作物の収穫量

多項式回帰モデルを採用する理由：畑に肥料をまくと作物の収穫量は増えますが，ある一定量を超えてまきすぎると，逆に土壌に悪影響を与え，収穫量が減少に転じることがあります．このような「山なり」の曲線的な関係を捉えるには，直線を仮定する単回帰モデルでは不十分であり，多項式モデルの方が適しています．

In [ ]:
#@title （準備）描画用の関数 plot_reg を定義します（理解不要）
#import matplotlib.pyplot as plt
!pip install japanize-matplotlib -q
import matplotlib.pyplot as plt
import japanize_matplotlib
from sklearn.preprocessing import PolynomialFeatures
import numpy as np

_size = 12
# 日本語フォントの設定
# グラフの文字サイズを大きめに設定 (具体的な数値を指定)
plt.rcParams['font.size'] = int(_size*1.2) # 全体のフォントサイズ
plt.rcParams['axes.labelsize'] = int(_size*1.4) # 軸ラベルのフォントサイズ
plt.rcParams['xtick.labelsize'] = int(_size) # x軸目盛りのフォントサイズ
plt.rcParams['ytick.labelsize'] = int(_size) # y軸目盛りのフォントサイズ
plt.rcParams['legend.fontsize'] = int(_size) # 凡例のフォントサイズ
plt.rcParams['figure.titlesize'] = int(_size*1.8) # 図全体のタイトルのフォントサイズ

npopts = lambda: np.printoptions(threshold=16,edgeitems=4)

def plot_reg(train, test=None, reg=None, degree=1, figsize=(5,4), s=int(_size*4), xlabel=None, ylabel=None, title=None, isGrid=True, savefig=None):
    x = train[0] if isinstance(train[0], np.ndarray) else train[0].values
    y = train[1] if isinstance(train[1], np.ndarray) else train[1].values
    xmin, xmax, dx = x.min(), x.max(), (x.max()-x.min())*0.2
    ymin, ymax, dy = y.min(), y.max(), (y.max()-y.min())*0.2

    if test is not None:
        xs = test[0] if isinstance(test[0], np.ndarray) else test[0].values
        ys = test[1] if isinstance(test[1], np.ndarray) else test[1].values
        xmin, xmax, dx = min(xmin, xs.min()), max(xmax, xs.max()), max(dx, (xs.max()-xs.min())*0.2)
        ymin, ymax, dy = min(ymin, ys[1].min()), max(ymax, ys[1].max()), max(dy, (ys[1].max()-ys[1].min())*0.54)

    #plt.figure(figsize=(3, 3.5))
    plt.figure(figsize=figsize)
    ax = plt.axes()
    ax.scatter(x, y, marker='o', c='darkviolet', s=s, zorder=10)
    if test is not None:
        ax.scatter(xs, ys, marker='x', c='0.75', s=s//2)
    if reg is not None:
        x_regr = np.linspace(xmin-dx, xmax+dx, 100)
        X_regr = PolynomialFeatures(degree).fit_transform(x_regr[:, np.newaxis])
        if isinstance(reg, (np.ndarray, list, tuple)):
            y_regr = X_regr.dot(np.asarray(reg))
        elif hasattr(reg, 'predict'):
            y_regr = reg.predict(X_regr) # モデルで予測値を計算
        else:
            y_regr = None # プロットしない
        if y_regr is not None:
            ax.plot(x_regr, y_regr, 'b-', zorder=20)

    ax.set_xlim(xmin-dx, xmax+dx)
    ax.set_ylim(ymin-dy, ymax+dy)
    plt.grid(isGrid)

    if xlabel is not None: plt.xlabel(xlabel)
    if ylabel is not None: plt.ylabel(ylabel)
    if title is not None: plt.title(title)

    plt.tight_layout()
    if savefig is not None:
        plt.savefig(savefig)


from IPython.display import display, Math
import sympy as sp
sp.init_printing()
def disp_residuals(y, X, beta, delta):
    ysp, Xsp, betasp, deltasp = sp.Matrix(y), sp.Matrix(X), sp.Matrix(beta), sp.Matrix(delta.round(1))
    yl, Xl, betal, deltal = sp.latex(ysp), sp.latex(Xsp), sp.latex(betasp), sp.latex(deltasp)
    eq = f"{deltal} = {yl} - {Xl} {betal}"
    display(Math(eq))


# https://chatgpt.com/s/t_6919c3ff01908191bc7edd10b9865da6
import numpy as np
from math import comb
def inverse_poly_coeffs(beta, x_mean, x_std, y_mean, y_std):
    """
    beta   : 標準化後の多項式回帰係数（長さ p+1）
    x_mean : x の平均
    x_std  : x の標準偏差
    y_mean : y の平均
    y_std  : y の標準偏差
    return : 標準化前の多項式回帰係数 α（長さ p+1）
    """
    p = len(beta) - 1
    alpha = np.zeros(p + 1)

    # k = 0（定数項）
    alpha[0] = y_mean + y_std * sum(
        beta[j] * ((-x_mean) ** j) / (x_std ** j) for j in range(p + 1)
    )

    # k = 1,...,p
    for k in range(1, p + 1):
        alpha[k] = y_std * sum(
            beta[j] * comb(j, k) * ((-x_mean) ** (j - k)) / (x_std ** j)
            for j in range(k, p + 1)
        )

    return alpha

In [ ]:
import pandas as pd

# データの準備
data = {
    '肥料の量 x (kg)': [10, 13, 17, 20, 23, 27, 30, 33, 37, 40, 43, 47, 50, 53, 57, 60],
    '収穫量 y (kg)': [479, 528, 554, 605, 602, 637, 650, 649, 673, 668, 675, 669, 672, 648, 611, 588]}
df = pd.DataFrame(data)

# データの表を表示
print("データ例：肥料の量と作物の収穫量")
print(df)

x_data = df['肥料の量 x (kg)'].values  # 説明変数 (特徴量)
y_data = df['収穫量 y (kg)'].values  # 目的変数

In [ ]:
plot_reg((x_data, y_data), xlabel=df.columns[0], ylabel=df.columns[1])

### 実習課題
1. `x_data` と `y_data` をそれぞれ標準化した `xs` と `ys` を作成せよ．
2. `xs` と `ys` の組から回帰用に $n=9$ 点をランダムに選出せよ．
3. 選出した $n$ 点を用いて，最小二乗法で多項式回帰を実行せよ．多項式の次数が $p=1$ から $8$ の結果を比較せよ．
4. $p$ 次多項式回帰（$p>8$，例：$p=10,20$）に$\ell_2$正則化を施せ．$\ell_2$ 正則化の重み $\lambda$ が $0.01$ から $100$ までの範囲で結果を比較せよ．
5. 残差平方和（RSS）を，選出しなかった点と比較せよ．

In [ ]:
# 1. x_data と y_data をそれぞれ標準化した xs と ys を作成する．
xs = (x_data - x_data.mean(axis=0)) / x_data.std(axis=0)
ys = (y_data - y_data.mean(axis=0)) / y_data.std(axis=0)
print(pd.DataFrame({'xs': xs, 'ys': ys}))

# 2. ランダムに n 点選出する．
np.random.seed(20251118)
n = 9
it = np.random.choice(len(x_data), size=n, replace=False)
ival = np.setdiff1d(np.arange(len(x_data)), it)
print("回帰用", it, ", 検証用", ival)

plot_reg((xs[it], ys[it]), (xs[ival], ys[ival]), xlabel='標準化済み x', ylabel='標準化済み y')

In [ ]:
# 3. xs[itrain] xs[itrain] を用いて多項式回帰を実行する．

p = 8 # 多項式の次数

# 計画行列 X を作成する．
X = np.column_stack([xs[it]**k for k in range(p + 1)])
with np.printoptions(precision=2, suppress=True):
    print("計画行列:\n", X)

# 4. ridge回帰を実行する（lam>0）
lam = 0    # 0.01, 0.1, 1, 10, 100
beta = np.linalg.inv(X.T.dot(X) + lam * np.eye(p+1)).dot(X.T.dot(ys[it]))

pd.set_option('display.float_format', '{:.3f}'.format)
print(pd.DataFrame({'beta': beta}).T)

plot_reg((xs[it], ys[it]), (xs[ival], ys[ival]), reg=beta, degree=p, xlabel='標準化済み x', ylabel='標準化済み y')

In [ ]:
# 5. RSSを比較する
ys_it = X.dot(beta)
rss = ((ys[it] - X.dot(beta))**2).sum()

# 検証：ys[ival] に対する予測値を多項式で xs[ival] から計算する
ys_ival = np.column_stack([xs[ival]**k for k in range(p + 1)]).dot(beta)
rss_val = ((ys[ival] - ys_ival)**2).sum()

print("回帰：RSS =", rss, "\n検証：RSS =", rss_val)

In [ ]:
# おまけ：標準化前の x の多項式の係数に換算する．
# https://chatgpt.com/s/t_6919c3ff01908191bc7edd10b9865da6
alpha = inverse_poly_coeffs(beta, x_data.mean(), x_data.std(), y_data.mean(), y_data.std())
plot_reg((x_data[it], y_data[it]), (x_data[ival], y_data[ival]), xlabel=df.columns[0], degree=p, reg=alpha, ylabel=df.columns[1])

---
## 【補遺】直交マッチング追跡（orthogonal matching pursuit; OMP）を実装します．

${\boldsymbol x}^\star=\arg\min_{\boldsymbol x}\|\boldsymbol x\|_0 \quad\mbox{subject to}\quad \|{\boldsymbol y}-{\boldsymbol{ Ax}}\|_2\leq \delta$

を数値計算するスパース解法のひとつです．
- [[Pati+93]](https://user.eng.umd.edu/~krishna/images/pati_reza_psk.pdf)
- 酒井の解説資料：[スライド](https://github.com/tsakailab/HDsci-SpM/blob/main/seminar/slides/Day03.pdf)，[テキスト](https://github.com/tsakailab/prml/blob/master/lectures/OMP_tsakai.pdf)
- 非ゼロ成分の計算は，残差が `delta` 以下または非ゼロが `maxnnz` 個以上になると終了します．

In [ ]:
import numpy as np
from numpy import linalg

# Orthogonal matching pursuit (OMP)
def OMP(A, y, delta=1e-6, maxnnz=None):
    d, n = A.shape
    if maxnnz is None: maxnnz = d // 2
    r = y.copy()
    x = np.zeros(n)
    supp = []                                   # 台（b の説明に使う A の列の番号のリスト）
    norma = np.linalg.norm(A, axis=0, keepdims=True)
    while len(supp) < maxnnz and linalg.norm(r) > delta:
        t = np.argmax(np.abs( A.T.dot(r) / norma))    # 残差 r と最も類似する行列 A の列の番号 t
        supp.append(t)                          # 台 supp に t を追加する
        Asupp = A[:,supp]                       # b の説明に用いる A の列ベクトル
        #### x[supp] = '''説明変数を Asupp，目的変数を b とする回帰係数の最小二乗解 '''
        x[supp] = linalg.lstsq(Asupp, y, rcond=None)[0]
        r = y - Asupp.dot(x[supp])              # 残差の更新
    return x